In [1]:
import os
import glob
import math
import random
from functools import reduce
from collections import Counter
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    GridSearchCV, 
    train_test_split, 
    KFold
)
from sklearn.preprocessing import (
    MinMaxScaler, 
    StandardScaler, 
    normalize
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    silhouette_score
)
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from skopt import gp_minimize, BayesSearchCV
from skopt.space import Real, Integer
from rdkit import Chem, DataStructs, RDLogger, RDConfig
from rdkit.Chem import (
    AllChem,
    MACCSkeys,
    ChemicalFeatures,
    Lipinski,
    rdmolops,
    MolStandardize
)
from rdkit.Chem.EState import Fingerprinter
from rdkit.Chem.AtomPairs import Torsions
from rdkit.Chem.Descriptors import rdMolDescriptors
RDLogger.DisableLog('rdApp.*')

In [2]:
def get_fps(smi_list):  # calculate molecular representation for model construction
    fps = []
    for i in range(len(smi_list)):
        mol = Chem.MolFromSmiles(MolStandardize.standardize_smiles(smi_list[i]))
        fps1 = AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=2048)
        fps.append(fps1)
    fps = np.array(fps)
    return fps

def calc_avg_1tanimoto(fingerprints_array) :        # calculate the structural sparsity
    n_samples = fingerprints_array.shape[0]
    avg_dissim = np.zeros(n_samples)
    for i in range(n_samples):
        fp_i = DataStructs.ExplicitBitVect(len(fingerprints_array[i]))
        for idx, bit in enumerate(fingerprints_array[i]):
            if bit:
                fp_i.SetBit(idx)
        sims = []
        for j in range(n_samples):
            if i != j:
                fp_j = DataStructs.ExplicitBitVect(len(fingerprints_array[j]))
                for idx, bit in enumerate(fingerprints_array[j]):
                    if bit:
                        fp_j.SetBit(idx)
                tanimoto = DataStructs.TanimotoSimilarity(fp_i, fp_j)
                sims.append(1 - tanimoto)
        avg_dissim[i] = np.mean(sims)
    return avg_dissim

def calc_avg_tanimoto(maccs_array):                 # calcualte the tanimoto similarity for uniform K-fold
    n_samples = maccs_array.shape[0]
    avg_dissim = np.zeros(n_samples)
    for i in range(n_samples):
        fp_i = DataStructs.ExplicitBitVect(len(maccs_array[i]))
        for idx, bit in enumerate(maccs_array[i]):
            if bit:
                fp_i.SetBit(idx)
        sims = []
        for j in range(n_samples):
            if i != j:
                fp_j = DataStructs.ExplicitBitVect(len(maccs_array[j]))
                for idx, bit in enumerate(maccs_array[j]):
                    if bit:
                        fp_j.SetBit(idx)
                tanimoto = DataStructs.TanimotoSimilarity(fp_i, fp_j)
                sims.append(tanimoto)
        avg_dissim[i] = np.mean(sims)
    return avg_dissim

def calculate_knn_density(labels, k=5):
    labels_2d = labels.reshape(-1, 1)
    n = len(labels_2d)
    if k >= n:
        raise ValueError("k needs to be smaller than n_sample_size")
    knn = NearestNeighbors(n_neighbors=k + 1)
    knn.fit(labels_2d)
    distances, _ = knn.kneighbors(labels_2d)
    r_k = distances[:, k]
    epsilon = 1e-8
    r_k_safe = np.where(r_k == 0, epsilon, r_k)
    density = k / (2 * r_k_safe)
    return density

def calculate_2_knn_density(fold_test_labels, fold_labels, k=5):
    fold_test_reshaped = fold_test_labels.reshape(-1, 1)
    fold_labels_reshaped = fold_labels.reshape(-1, 1)
    if k >= len(fold_labels):
        raise ValueError(f"k needs to be smaller than n_sample_size")
    knn = NearestNeighbors(n_neighbors=k + 1)
    knn.fit(fold_labels_reshaped)
    distances, _ = knn.kneighbors(fold_test_reshaped)
    r_k = distances[:, k]
    epsilon = 1e-8
    r_k_safe = np.where(r_k == 0, epsilon, r_k)
    density = k / (2 * r_k_safe)
    return density

def get_model_index(labels,fingerprints):
    reg = RandomForestRegressor(n_jobs=60,random_state=100)
    scaler = StandardScaler()
    scaler = scaler.fit(labels.reshape(-1,1))
    normed_labels = scaler.transform(labels.reshape(-1,1))
    reg  = reg.fit(fingerprints,normed_labels.ravel())
    predictions = scaler.inverse_transform(reg.predict(fingerprints).reshape(-1,1))
    mse = (predictions-labels.reshape(-1,1))**2
    EDr2 = mean_squared_error(labels,predictions)
    n = fingerprints.shape[0]
    return reg,predictions,mse,EDr2,n
    
def get_aboslute_indices(save_data,data):
    if save_data.shape[0] == 0:
        return []
    else:
        save_col1 = np.array(save_data)[:, 0]
        data_col1 = np.array(data)[:, 0]
        mask = np.isin(data_col1,save_col1)
        indices = np.where(mask)[0].tolist()
        return indices

def uniform_five_fold_cv(arr):
    n_samples = arr.shape[0]
    indices = np.arange(n_samples)
    for i in range(5):
        test_indices = indices[i::5]
        train_indices = np.setdiff1d(indices, test_indices)
        yield train_indices, test_indices

def get_multi_fps(smi_list,type):
    if type == 1 :
        fps = []
        for i in range(len(smi_list)):
            mol = Chem.MolFromSmiles(MolStandardize.standardize_smiles(smi_list[i]))
            fps0 = AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=2048)
            fps.append(fps0)
        fps = np.array(fps)
    elif type == 2:
        fps = []
        for i in range(len(smi_list)):
            mol = Chem.MolFromSmiles(MolStandardize.standardize_smiles(smi_list[i]))
            maccs = MACCSkeys.GenMACCSKeys(mol)
            fps2 = AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=2048)
            fps0 = AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=2048)
            fps.append(fps0+fps2+maccs)
        fps = np.array(fps)
    elif type == 3:
        fps = []
        for i in range(len(smi_list)):
            mol = Chem.MolFromSmiles(MolStandardize.standardize_smiles(smi_list[i]))
            maccs = MACCSkeys.GenMACCSKeys(mol)
            fps2 = AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=2048)
            fps0 = AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=2048)
            fps6 = Chem.RDKFingerprint(mol,minPath=2,maxPath=2,fpSize=2048)
            fps.append(fps0+fps2+maccs+fps6)
        fps = np.array(fps)
    elif type == 4:
        fps = []
        estate = []
        for i in range(len(smi_list)):
            mol = Chem.MolFromSmiles(MolStandardize.standardize_smiles(smi_list[i]))
            maccs = MACCSkeys.GenMACCSKeys(mol)
            fps2 = AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=2048)
            fps0 = AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=2048)
            fps6 = Chem.RDKFingerprint(mol,minPath=2,maxPath=2,fpSize=2048)
            fps5 = Fingerprinter.FingerprintMol(mol)[0]
            estate.append(fps5)
            fps.append(fps0+fps2+maccs+fps6)
        fps = np.array(fps)
        fps = np.concatenate([fps,np.vstack(estate)],axis=1)
    return fps

def get_fps_uncertainty(data_remaining):
    all_train_uncertainty_record = []
    all_test_uncertainty_record = []
    for f in [1,2,3,4]:
        data_remaining.columns = ['SMILES','Label']
        smiles = data_remaining['SMILES'].tolist()
        labels = data_remaining['Label'].values
        fingerprints = get_multi_fps(smiles,f)
        x_train = fingerprints
        train_labels = labels.reshape(-1,1)
        train_scaler = StandardScaler().fit(train_labels)
        normed_train = train_scaler.transform(train_labels)
        x_test = x_train
        reg = RandomForestRegressor(n_jobs=-1,random_state=42)
        reg  = reg.fit(x_train,normed_train.ravel())
        test_predictions = np.zeros((x_test.shape[0], len(reg.estimators_)))
        for i, tree in enumerate(reg.estimators_):
            tree_pred = tree.predict(x_test)
            test_predictions[:, i] = tree_pred
        test_uncertainty = np.std(test_predictions,axis=1).reshape(-1,1)
        all_test_uncertainty_record.append(test_uncertainty)
    return all_train_uncertainty_record,all_test_uncertainty_record
    
def merge_arrays(data):
    arrays = [item for item in data]
    merged_array = np.hstack(arrays)
    return merged_array

def get_bootstrap(fold_fingerprints,fold_labels,x_test):
    test_uncertainty_record = []
    n_samples = len(fold_fingerprints)
    n_subsets = 9 
    bootstrap_indices = []
    for _ in range(n_subsets):
        indices = np.random.choice(n_samples, size=int(n_samples*0.9), replace=True)
        bootstrap_indices.append(indices)
    for i in range(0,n_subsets):
        globals()[f'bootstrap_fingerprints{i}'] = fold_fingerprints[bootstrap_indices[i]]
        globals()[f'bootstrap_labels{i}'] = fold_labels[bootstrap_indices[i]]
        boot_reg = RandomForestRegressor(n_jobs=-1,random_state=100)
        boot_scaler = StandardScaler().fit(globals()[f'bootstrap_labels{i}'].reshape(-1,1))
        boot_normed_labels = boot_scaler.transform(globals()[f'bootstrap_labels{i}'].reshape(-1,1))
        boot_reg  = boot_reg.fit(globals()[f'bootstrap_fingerprints{i}'],boot_normed_labels.ravel())
        test_predictions = np.zeros((x_test.shape[0], len(boot_reg.estimators_)))
        for a, tree in enumerate(boot_reg.estimators_):
            tree_pred = tree.predict(x_test)
            test_predictions[:, a] = tree_pred
        test_uncertainty = np.std(test_predictions,axis=1).reshape(-1,1)
        test_uncertainty_record.append(test_uncertainty)
    return np.std(np.array(test_uncertainty_record),axis=0)


def get_v(fold_data,fold_fingerprints,fold_labels,fold_test_fingerprints):
    _,all_val_uncertainty_record = get_fps_uncertainty(fold_data)
    fps_std_uncertainty_record = merge_arrays(all_val_uncertainty_record)
    fps_std = np.std(fps_std_uncertainty_record,axis=1)
    bootstrap_uncertainty = get_bootstrap(fold_fingerprints,fold_labels,fold_test_fingerprints)
    return bootstrap_uncertainty.reshape(-1)**2/(math.e**fps_std.reshape(-1))

In [3]:
path = os.getcwd()
data = pd.read_csv(f'{path}/datasets/example_train.csv').iloc[:,:2]         # please check molecular data file. It should be in SMILES-Labels formatting, which contains two columns.
data.columns = ['SMILES','Label']
origin_smiles = data['SMILES'].tolist()
origin_labels = data['Label'].values
data_labels = origin_labels.copy()

origin_fingerprints = get_fps(origin_smiles)
fingerprints = origin_fingerprints.copy()

origin_labels_density = calculate_knn_density(origin_labels)
labels_density = origin_labels_density.copy()

tanimoto = calc_avg_1tanimoto(origin_fingerprints).tolist()
tanimoto_copy = np.array(tanimoto.copy())
sort_indices = np.argsort(tanimoto)

origin_tanimoto = np.array([tanimoto[i] for i in sort_indices])
origin_fingerprints = np.array([origin_fingerprints[i] for i in sort_indices])
origin_labels = np.array([origin_labels[i] for i in sort_indices])
origin_labels_density = np.array([origin_labels_density[i] for i in sort_indices])
origin_data = data.iloc[sort_indices].reset_index(drop=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_train_clean_data_list, fold_test_clean_data_list = [] , []
fold_train_filtered_data_list , fold_test_filtered_data_list = [] , []
clean_indices_list, filtered_indices_list = [], []

param_space = {
    'n_estimators': Integer(50, 200),
    'max_depth': Integer(3, 10),
    'max_features': ['sqrt', 'log2']
}

for fold, (train_index, test_index) in enumerate(uniform_five_fold_cv(origin_fingerprints)):        # test equals to val in K-fold
    train_index,test_index = list(train_index),list(test_index)
    fold_fingerprints, fold_test_fingerprints = origin_fingerprints[train_index], origin_fingerprints[test_index]
    fold_labels, fold_test_labels = origin_labels[train_index], origin_labels[test_index]

    fold_tanimoto_1,fold_test_tanimoto_1 = origin_tanimoto[train_index],origin_tanimoto[test_index]
    fold_density,fold_test_density = origin_labels_density[train_index],origin_labels_density[test_index]
    fold_train_data,fold_test_data = origin_data.iloc[train_index],origin_data.iloc[test_index]
    
    train_clean_indices = [i for i in range(0,fold_train_data.shape[0])]
    test_clean_indices = [i for i in range(0,fold_test_data.shape[0])]
    train_filtered_indices,test_filtered_indices = [0] , [0]

    fold_fingerprints, fold_test_fingerprints = fold_fingerprints[train_clean_indices], fold_test_fingerprints[test_clean_indices]
    fold_labels, fold_test_labels = fold_labels[train_clean_indices], fold_test_labels[test_clean_indices]
    fold_train_data,fold_test_data = fold_train_data.iloc[train_clean_indices,:],fold_test_data.iloc[test_clean_indices,:]

    fold_v = get_v(fold_train_data,fold_fingerprints,fold_labels,fold_fingerprints)
    fold_v = fold_v.reshape(-1,1)
    
    fold_tanimoto_1 = MinMaxScaler().fit_transform(fold_tanimoto_1.reshape(-1,1))
    fold_test_tanimoto_1 = MinMaxScaler().fit_transform(fold_test_tanimoto_1.reshape(-1,1))
    fold_density = MinMaxScaler().fit_transform(fold_density.reshape(-1,1))
    fold_test_density = MinMaxScaler().fit_transform(fold_test_density.reshape(-1,1))
    
    silhouette_scores = []
    for k in range(2, 20):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(fold_fingerprints)
        silhouette_scores.append(silhouette_score(fold_fingerprints, labels))
    best_k = range(2, 20)[np.argmax(silhouette_scores)]
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(fold_fingerprints)

    whole_predictions_list = []
    for i in range(best_k):
        idx = np.where(cluster_labels == i)[0]
        fingerprints_i = fold_fingerprints[idx]
        labels_i = fold_labels[idx]
        reg_i, preds_i, mse, r2, n = get_model_index(labels_i, fingerprints_i)
        preds_test = reg_i.predict(fold_test_fingerprints).reshape(-1, 1)
        preds_test = StandardScaler().fit_transform(preds_test)
        whole_predictions_list.append(preds_test)
    stacked = np.hstack(whole_predictions_list)
    h = np.mean(np.std(stacked, axis=1))

    scaler = StandardScaler().fit(fold_labels.reshape(-1,1))
    normed_labels = scaler.transform(fold_labels.reshape(-1,1))
    
    opt = BayesSearchCV(
        estimator=RandomForestRegressor(random_state=100, n_jobs=60),
        search_spaces=param_space,
        n_iter=8,
        n_jobs=60,
        scoring='neg_mean_absolute_error',
        random_state=42,
        verbose=0
    )

    opt.fit(fold_fingerprints,normed_labels.ravel())
    
    best_params = opt.best_params_
    reg = RandomForestRegressor(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        max_features=best_params['max_features'],
        n_jobs=60,
        random_state=100
    )
    
    reg = reg.fit(fold_fingerprints,normed_labels.ravel())  

    train_predictions = scaler.inverse_transform(reg.predict(fold_fingerprints).reshape(-1,1))
    train_EDr2 = mean_squared_error(fold_labels,train_predictions)     
    train_n = fold_fingerprints.shape[0]
    train_bias = fold_labels.reshape(-1,1) -train_predictions.reshape(-1,1)
    uncertainty_predictions = np.zeros((fold_fingerprints.shape[0], len(reg.estimators_)))
    for i, tree in enumerate(reg.estimators_):
        tree_pred = tree.predict(fold_fingerprints)
        uncertainty_predictions[:, i] = tree_pred
    train_uncertainty = np.std(uncertainty_predictions,axis=1).reshape(-1,1)

    test_predictions = scaler.inverse_transform(reg.predict(fold_test_fingerprints).reshape(-1,1))
    test_EDr2 = mean_squared_error(fold_test_labels,test_predictions)
    test_n = fold_test_fingerprints.shape[0]
    test_bias = fold_test_labels.reshape(-1,1) - test_predictions.reshape(-1,1)
    uncertainty_predictions = np.zeros((fold_test_fingerprints.shape[0], len(reg.estimators_)))
    for i, tree in enumerate(reg.estimators_):
        tree_pred = tree.predict(fold_test_fingerprints)
        uncertainty_predictions[:, i] = tree_pred
    test_uncertainty = np.std(uncertainty_predictions,axis=1).reshape(-1,1)
    
    train_label_density = calculate_knn_density(fold_labels, k=5)
    train_label_density = ((train_label_density - train_label_density.min()) / (train_label_density.max() - train_label_density.min())).tolist()

    def objective(filter_ratio_list, \
                    train_uncertainty= train_uncertainty, \
                        train_fingerprints=  fold_fingerprints, \
                            test_fingerprints = fold_test_fingerprints, \
                                train_labels = fold_labels, \
                                    test_labels = fold_test_labels):
        
        train_filter_ratio = filter_ratio_list[0]
        eta = 0.95
        train_evaluator = (train_bias**2*train_uncertainty-fold_v).reshape(-1)
        
        train_sorted_indices = np.argsort(train_evaluator)[::-1]
        train_num_to_filter = int(len(fold_labels) * train_filter_ratio)
        train_filtered_indices = train_sorted_indices[:train_num_to_filter]
        train_clean_indices = train_sorted_indices[train_num_to_filter:]

        train_filtered_indices = list(set(train_filtered_indices))
        origin_list = list(range(len(train_uncertainty)))
        train_clean_indices = list(set(origin_list) - set(train_filtered_indices))

        train_clean_fingerprints = train_fingerprints[train_clean_indices]
        train_clean_labels = train_labels[train_clean_indices]
        
        test_clean_fingerprints = test_fingerprints[test_clean_indices]
        test_clean_labels = test_labels[test_clean_indices]

        clean_scaler = StandardScaler().fit(train_clean_labels.reshape(-1, 1))
        normed_train_clean_labels = clean_scaler.transform(train_clean_labels.reshape(-1, 1))

        opt = BayesSearchCV(
            estimator=RandomForestRegressor(random_state=100, n_jobs=60),
            search_spaces=param_space,
            n_iter=8,
            n_jobs=60,
            scoring='neg_mean_absolute_error',
            random_state=42,
            verbose=0
        )

        opt.fit(train_clean_fingerprints, normed_train_clean_labels.ravel())
        
        best_params = opt.best_params_
        clean_reg = RandomForestRegressor(
            n_estimators=best_params['n_estimators'],
            max_depth=best_params['max_depth'],
            max_features=best_params['max_features'],
            n_jobs=60,
            random_state=100
        )
        
        clean_reg = clean_reg.fit(train_clean_fingerprints, normed_train_clean_labels.ravel())
        train_clean_predictions = clean_scaler.inverse_transform(clean_reg.predict(train_clean_fingerprints).reshape(-1, 1))
        train_EDFr2 = mean_squared_error(train_clean_labels, train_clean_predictions)
        train_MSE = train_EDFr2 / train_EDr2
        train_nF = train_clean_fingerprints.shape[0]
        

        train_epsilon_D = 1 / (1 - ((h * (math.log(train_n / h) + 1) - math.log(eta)) / train_n) ** 0.5)
        train_epsilon_DF = 1 / (1 - ((h * (math.log(train_nF / h) + 1) - math.log(eta)) / train_nF) ** 0.5)
        train_betaT = train_epsilon_D / train_epsilon_DF
        train_target = (train_betaT - train_MSE) * train_epsilon_DF
        
        test_clean_predictions = clean_scaler.inverse_transform(clean_reg.predict(test_clean_fingerprints).reshape(-1, 1))
        test_EDFr2 = mean_squared_error(test_clean_labels, test_clean_predictions)
        target = train_target *(1/math.e**test_EDFr2)


        return -target

    # Here, usrs can choose the filtering lower and upper limits for each round of filtering to control the intensity.
    space = [Real(0.0, 0.5, name='train_filter_ratio')]          # noisy_sample_ratio (0,1] we recommond that it should use a small value. In practical, the user may need to try different upper limit

    result = gp_minimize(
        objective,
        space,
        n_calls=10,
        random_state=42,
        n_random_starts=5
    )

    train_best_ratio = result.x[0]
    best_target = -result.fun
    train_evaluator = (train_bias**2*train_uncertainty-fold_v).reshape(-1)
    train_sorted_indices = np.argsort(train_evaluator)[::-1]
    train_num_to_filter = int(len(fold_labels) * train_best_ratio)
    train_filtered_indices = train_sorted_indices[:train_num_to_filter]
    
    train_filtered_indices = list(set(train_filtered_indices))
    origin_list = list(range(len(train_uncertainty)))
    train_clean_indices = list(set(origin_list) - set(train_filtered_indices))
    
    fold_train_clean_data_list.append(fold_train_data.iloc[train_clean_indices,:])
    fold_train_filtered_data_list.append(fold_train_data.iloc[train_filtered_indices,:])
    
    
best_count = None
best_r2 = float('-inf')
best_params = None

# 4 indicates no filtering ,and it can also proved users with a reference to see if no filtering yields the best results.
for count in [0,1,2,3,4]:  ## as described in our paper, using a quit conservative count (3) often produces the best model performance. In practical, the user also can try different intensity
    
    arr = np.concatenate(fold_train_filtered_data_list, axis=0)
    df_arr = pd.DataFrame(arr)
    
    value_counts = df_arr.value_counts().reset_index(name='count')
    save_filtered_data = value_counts[value_counts['count'] > count].drop(columns='count').to_numpy()
    
    filtered_indices = get_aboslute_indices(save_filtered_data, data)
    clean_indices = list(set(range(data.shape[0])) - set(filtered_indices))
    x_train = fingerprints[clean_indices]
    train_labels = data_labels[clean_indices].reshape(-1, 1)
    scaler = StandardScaler().fit(train_labels)
    normed_labels = scaler.transform(train_labels).ravel()
    x_train_split, x_val_split, y_train_split, y_val_split = train_test_split(
        x_train, normed_labels, test_size=0.1, random_state=42)

    opt = BayesSearchCV(
        estimator=RandomForestRegressor(random_state=100, n_jobs=-1),
        search_spaces=param_space,
        n_iter=8,
        n_jobs=60,
        scoring='neg_mean_absolute_error',
        random_state=42,
        verbose=0
    )

    opt.fit(x_train_split, y_train_split)
    
    best_model = opt.best_estimator_
    y_pred = best_model.predict(x_val_split)
    best_r2 = r2_score(y_val_split, y_pred)

        
    arr = np.concatenate(fold_train_filtered_data_list)
    save_filtered_data = pd.DataFrame(arr).value_counts().reset_index(name='count')[pd.DataFrame(arr).value_counts().reset_index(name='count')['count'] > count].drop(columns='count').to_numpy()
    filtered_indices = get_aboslute_indices(save_filtered_data,data)
    clean_indices = list(set(list(range(data.shape[0]))) - set(filtered_indices))
    save_clean_data = data.iloc[clean_indices,:]
    
    if len(filtered_indices) !=0:
        noisy_train = fingerprints[filtered_indices]
        noisy_labels = data_labels[filtered_indices].reshape(-1,1)
        noise_tanimoto = tanimoto_copy[filtered_indices]
        noise_labels_density = labels_density[filtered_indices]
        noise_ag_index = MinMaxScaler().fit_transform(MinMaxScaler().fit_transform(noise_labels_density.reshape(-1,1)-noise_tanimoto.reshape(-1,1)))
        x_train = fingerprints[clean_indices]
        train_labels = data_labels[clean_indices].reshape(-1,1)

        x_temp, x_val, y_temp, y_val = train_test_split(
            x_train, train_labels, test_size=0.1, random_state=42
        )


        sorted_indices = np.argsort(noise_ag_index.flatten())
        sorted_noisy_train = noisy_train[sorted_indices]
        sorted_noisy_labels = noisy_labels[sorted_indices]
        sorted_filtered_indices = list(np.array(filtered_indices)[sorted_indices])

        best_score = float('inf')
        best_num_noise = 0
        best_model = None
        all_scores = []
        max_noise_to_try = max(1, len(noisy_train))
        r2_results = {}
        def objective(num_noise_list):
            num_noise = int(num_noise_list[0])
            if num_noise == 0:
                x_train_final = x_temp
                y_train_final = y_temp
            else:
                x_train_final = np.vstack([x_temp, sorted_noisy_train[:num_noise]])
                y_train_final = np.vstack([y_temp, sorted_noisy_labels[:num_noise]])

            scaler = StandardScaler()
            y_train_final_scaled = scaler.fit_transform(y_train_final).ravel()
            y_val_scaled = scaler.transform(y_val)
            
            opt = BayesSearchCV(
                estimator=RandomForestRegressor(random_state=100, n_jobs=60),
                search_spaces=param_space,
                n_iter=8,
                n_jobs=60,
                scoring='neg_mean_absolute_error',
                random_state=42,
                verbose=0
            )
            opt.fit(x_train_final, y_train_final_scaled)

            best_params = opt.best_params_

            model = RandomForestRegressor(
                n_estimators=best_params['n_estimators'],
                max_depth=best_params['max_depth'],
                max_features=best_params['max_features'],
                n_jobs=60,
                random_state=100
            )
            model.fit(x_train_final, y_train_final_scaled)
            y_pred_scaled = model.predict(x_val)
            bias_squared = mean_squared_error(y_val_scaled, y_pred_scaled)
            individual_predictions = np.array([tree.predict(x_val) for tree in model.estimators_])
            uncertainty = np.std(individual_predictions, axis=0).mean()
            score = bias_squared + uncertainty
            r2 = r2_score(y_val_scaled, y_pred_scaled)
            r2_results[num_noise] = r2
            return score
        res = gp_minimize(objective,
                        [(0, max_noise_to_try)],
                        acq_func="EI",
                        n_calls=10,
                        random_state=42)

        best_num_noise = int(res.x[0])
        best_score = res.fun
        r2 = r2_results[best_num_noise]
        
        if r2 > best_r2:
            best_r2 = r2
            best_count = count
            best_params = opt.best_params_

        if best_num_noise == 0:
            final_train_set = x_temp
            final_train_labels = y_temp
        else:
            final_train_set = np.vstack([x_temp, sorted_noisy_train[:best_num_noise]])
            final_train_labels = np.vstack([y_temp, sorted_noisy_labels[:best_num_noise]])

        clean_indices = list(clean_indices) + list(sorted_filtered_indices[:best_num_noise])
        filtered_indices = list(set(filtered_indices) - set(sorted_filtered_indices[:best_num_noise]))
        
        save_clean_data = data.iloc[clean_indices,:]
        save_clean_data.columns=['SMILES','Labels']
        save_clean_data.to_csv(f'{path}/datasets/results/clean_{count}_{best_count}.csv',index=False)
        save_filtered_data = data.iloc[filtered_indices,:]
        save_filtered_data.columns=['SMILES','Labels']
        save_filtered_data.to_csv(f'{path}/datasets/results/noise_{count}_{best_count}.csv',index=False)
        
    else:
        save_clean_data = data.iloc[clean_indices,:]
        save_clean_data.columns=['SMILES','Labels']
        save_clean_data.to_csv(f'{path}/datasets/results/clean_{count}_{best_count}.csv',index=False)
        save_filtered_data = data.iloc[filtered_indices,:]
        save_filtered_data.columns=['SMILES','Labels']
        save_filtered_data.to_csv(f'{path}/datasets/results/noise_{count}_{best_count}.csv',index=False)